# Data

In [1]:
from typing import TypedDict,Annotated
from pydantic import BaseModel,Field
import operator

In [2]:
class DiscussionMessage(BaseModel):

    round: int

    persona_id: str

    persona_name: str

    content: str


class ChatMessage(BaseModel):
    role: str
    content: str


class PersonaDefinition(BaseModel):
    id: str
    name: str
    age: int
    gender:str
    behavior: str
    expertise: str
    goal: str
    personality: str
    speaking_style: str

class PersonasResponse(BaseModel):
    personas: list[PersonaDefinition]

class RuntimePersona(PersonaDefinition):
    model: str | None = None
    chat_history: list[ChatMessage] = Field(default_factory=list)

In [3]:
from copy import deepcopy

def merge_personas(old: dict[str, RuntimePersona],new: dict[str, RuntimePersona]) -> dict[str, RuntimePersona]:

    merged = deepcopy(old)

    for persona_id, update in new.items():

        if persona_id not in merged:
            merged[persona_id] = update
            continue
        persona = merged[persona_id]

        # This is For When A model Is Provided in The Worker Node It Will Updated Here
        if update.model is not None:
            persona.model = update.model

        # It will Add or Append The new Messages if Existed
        if update.chat_history:
            persona.chat_history.extend(update.chat_history)

    return merged


In [4]:
class State(TypedDict):

    query: str

    num_personas: int
    
    personas: Annotated[
        dict[str, RuntimePersona],
        merge_personas
    ]

    discussion: Annotated[
        list[DiscussionMessage],
        operator.add
    ]

    review_summary: str

    critic_feedback: str

    final_answer: str

    final_discussion_summary: str

    round: int

    max_rounds: int

    judge_approved: bool

    judge_reason: str

    # available_models:list[str]

In [5]:
class JudgeDecision(BaseModel):
    judge_approved: bool
    judge_reason: str

# Graph

## Persona Planner/Creator

In [6]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

In [7]:
llm_planner=init_chat_model(
    model="qwen2.5:3b",
    model_provider="ollama"
)
llm_planner_struct= llm_planner.with_structured_output(PersonasResponse)
planner_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
                You are an expert AI Debate Planner.

                Your task is to design a team of expert personas that will collaboratively analyze and debate the user's problem from multiple perspectives.

                Instructions:

                1. Generate EXACTLY {num_personas} personas.
                2. Select personas that are directly relevant to solving the user's problem.
                3. Every persona must contribute a unique perspective and expertise.
                4. Avoid duplicate or highly similar roles.
                5. The combined panel should provide comprehensive coverage of the problem.
                6. Each generated persona must populate every field required by the output schema with meaningful and contextually relevant values.
                7. Prefer realistic and professional expert roles whenever appropriate.
                8. If multiple experts belong to the same domain, ensure they have clearly different objectives, reasoning styles, or viewpoints.
                9. The personas should be capable of constructive disagreement when appropriate instead of reaching identical conclusions.
                10. Ensure diversity in:
                - Expertise
                - Decision-making style
                - Priorities
                - Risk tolerance
                - Perspective
                11. Prioritize relevance over diversity. Every persona must have a meaningful contribution toward solving the user's problem.
                12. Each persona should be able to independently analyze the problem, defend its viewpoint with logical reasoning, respectfully challenge other viewpoints when appropriate, and adapt its opinion if presented with stronger evidence during later discussion rounds.
                13. Return ONLY the structured output matching the provided schema.
                14. Do NOT include explanations, markdown, code fences, or any additional text outside the structured output.
            """
        ),
        (
            "human",
            """
                User Query:
                {query}

                Required Number of Personas:
                {num_personas}

                Design the highest-quality expert panel possible for this problem.
            """
        ),
    ]
)
def persona_planner(state:State):
    messages = planner_prompt.format_messages(
        query=state["query"],
        num_personas=state["num_personas"],
    )

    try:
        response=llm_planner_struct.invoke(messages)
    except Exception as e:
        raise RuntimeError(f"Personas Generation Failed: {e}")
    runtime_personas={}
    for i, persona in enumerate(response.personas, start=1):
        runtime_personas[f"persona{i}"] = RuntimePersona(
        id=f"persona{i}",
        **persona.model_dump(exclude={"id"})
    )
    return {
        "personas": runtime_personas
    }

## Dispatch/Run Parallel Agents

In [8]:
from langgraph.types import Send

C:\Users\mvnsu\AppData\Roaming\Python\Python311\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [9]:
def dispatcher(state:State):
    personas=state["personas"]
    sends=[]
    for persona in personas.values():
        sends.append(
            Send(
                "worker",
                {
                    **state,
                    "current_persona":persona
                }
            )
        )
    
    return sends


## Agent/Worker Node

In [10]:
from langchain_core.prompts import ChatPromptTemplate
import random
from langchain.chat_models import init_chat_model
import json

### Models

In [11]:
MODELS={
    "qwen2.5:3b":init_chat_model(
        model="qwen2.5:3b",
        model_provider="ollama"
    ),

    "phi4-mini":init_chat_model(
    model="phi4-mini",
    model_provider="ollama"
    )
}
AVAILABLE_MODELS = list(MODELS.keys())

In [12]:
worker_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
                You are participating in a multi-agent discussion.

                You must always respond according to your assigned persona.

                Persona Information

                Name: {name}

                Age: {age}

                Gender:{gender}

                Expertise: {expertise}

                Behavior: {behavior}

                Personality: {personality}

                Speaking Style: {speaking_style}

                Goal: {goal}

                Current Round:
                {round}

                Review Summary:
                {review_summary}

                Critic Feedback:
                {critic_feedback}

                Discussion So Far (JSON):
                {discussion}

                Your Previous Messages (JSON):
                {chat_history}

                Interpret these JSON arrays as the discussion history and your previous responses.

                Each object represents one message.

                Use these histories to understand previous reasoning before generating your next response.
                
                The discussion is ordered chronologically from oldest to newest.
                
                The latest messages represent the most recent state of the discussion.

                Instructions:

                - Keep your response concise (around 100–400 words) unless more detail is necessary.
                - Stay consistent with your assigned persona throughout the discussion.
                - Use your expertise and professional perspective to guide your reasoning and support your response.
                - Build upon the existing discussion instead of starting from scratch.
                - Contribute new insights whenever possible.
                - Do not repeat your previous arguments verbatim unless reinforcement is necessary.
                - If another persona presents a stronger argument, you may revise your opinion and clearly explain why.
                - Do not agree simply for the sake of agreement. Respectfully challenge viewpoints when you disagree, supporting your reasoning with evidence or logical arguments.
                - Consider the reviewer summary and critic feedback when refining your response.
                - Focus on helping the group reach the most accurate and well-reasoned conclusion.
                - Respond naturally as your assigned persona.
                - Give one thoughtful response.
            """
        ),
        (
            "human",
            """
                User Query:

                {query}
            """
        )
    ]
)

def worker(state:State):
    persona=state["current_persona"]
    model=random.choice(AVAILABLE_MODELS)
    llm=MODELS[model]
    messages=worker_prompt.format_messages(

        query=state["query"],

        name=persona.name,

        age=persona.age,

        gender=persona.gender,

        expertise=persona.expertise,

        behavior=persona.behavior,

        personality=persona.personality,

        speaking_style=persona.speaking_style,

        goal=persona.goal,

        round=state["round"],

        review_summary=state["review_summary"],

        critic_feedback=state["critic_feedback"],

        discussion=json.dumps([msg.model_dump() for msg in state["discussion"]],indent=2,ensure_ascii=False),

        chat_history=json.dumps([msg.model_dump() for msg in persona.chat_history],indent=2,ensure_ascii=False)
        
    )

    try:
        response = llm.invoke(messages)
    except Exception as e:
        raise RuntimeError(
            f"Worker '{persona.name}' failed using model '{model}': {e}"
        )

    new_chat_insertion=ChatMessage(
        role=persona.name,
        content=response.content
    )

    updated_persona=RuntimePersona(
        **persona.model_dump(exclude={"model","chat_history"}),
        model=model,
        chat_history=[new_chat_insertion]
    )

    discussion_msg=DiscussionMessage(
        round=state["round"],
        persona_id=persona.id,
        persona_name=persona.name,
        content=response.content
    )

    return {
        "personas": {
            persona.id:updated_persona
        },
        "discussion": [
            discussion_msg
        ]
    }

## Reviewer

In [13]:
reviewer_prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """
                You are an impartial discussion reviewer.

                Your role is to objectively summarize the discussion.

                You must never introduce new opinions, assumptions, recommendations, or solutions.

                Your responsibilities:

                - Summarize the key arguments presented by each persona.
                - Identify major agreements.
                - Identify major disagreements.
                - Highlight important evidence or reasoning.
                - Mention unresolved questions or uncertainties.
                - Keep the summary concise and factual, and well organized.
                - Do not favor any persona.
                - Do not critique the discussion.
                - Do not provide a final decision.
                - Preserve the intent of each persona's arguments without changing their meaning.
                - Return only the review summary as plain text.
                - Do not use markdown, bullet points, JSON, XML, or code blocks.
                - Do not omit important arguments even if they are mentioned by only one persona.
            """
        ),

        (
            "human",
            """
                User Query:

                {query}

                Current Round:

                {round}

                Discussion (JSON):

                {discussion}
                
                Each JSON object represents one discussion message.

                The discussion is ordered chronologically from oldest to newest.

                Base your summary only on the discussion provided.

                Do not invent arguments or facts that are not present.
            """
        )

])
llm_reviewer=init_chat_model(
    model="phi4-mini",
    model_provider="ollama"
)
def reviewer(state:State):
    messages=reviewer_prompt.format_messages(
        query=state["query"],

        round=state["round"],

        discussion=json.dumps([msg.model_dump() for msg in state["discussion"]],indent=2,ensure_ascii=False)
    )

    try:

        response = llm_reviewer.invoke(messages)

    except Exception as e:

        raise RuntimeError(

            f"Reviewer failed: {e}"

        )
    
    return {

        "review_summary": response.content

    }

## Critic

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model

In [15]:
critic_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
            You are an expert discussion critic.

            Your responsibility is to improve the quality of the discussion.

            You must NEVER answer the user's question yourself.

            You must NEVER summarize the discussion.

            Instead, critically evaluate the discussion and identify how it can be improved.

            Responsibilities:

            - Identify weak or unsupported arguments.
            - Identify logical inconsistencies.
            - Identify missing perspectives or expertise.
            - Identify assumptions that require evidence.
            - Point out unanswered questions.
            - Detect if important risks or opportunities were ignored.
            - Detect repetitive or redundant arguments that do not meaningfully advance the discussion.
            - Suggest specific topics, assumptions, or questions that should be addressed in the next discussion round.
            - Be objective and constructive.
            - Do not criticize writing style.
            - Focus only on the quality, completeness, and logical consistency of the reasoning.
            - Your feedback will be provided to all discussion participants in the next round.
            - Write feedback that is specific, actionable, and focused on improving the quality of reasoning.
            - Do not introduce new solutions or arguments.
            - Only evaluate the quality of the existing discussion.
            - Use the reviewer summary as a high-level overview of the discussion and rely on the discussion history to verify details.
            - Prioritize feedback that will most improve the next discussion round.
            - Distinguish between disagreements caused by missing evidence and disagreements caused by different priorities or values.
            
            Return only constructive feedback as plain text that can be directly used by the discussion participants in the next round.

            Do not use markdown, bullet points, JSON, XML, or code blocks.
        """
    ),

    (
        "human",
        """
            User Query:

            {query}

            Current Round:

            {round}

            Reviewer Summary:

            {review_summary}

            Discussion (JSON):

            {discussion}

            Each JSON object represents one discussion message.

            The discussion is ordered chronologically.

            Evaluate the discussion and provide constructive feedback for improving the next discussion round.
        """
    )

])
llm_critic=init_chat_model(
    model="qwen2.5:3b",
    model_provider="ollama"
)
def critic(state:State):
    messages=critic_prompt.format_messages(

        query=state["query"],

        round=state["round"],

        review_summary=state["review_summary"],

        discussion=json.dumps([msg.model_dump() for msg in state["discussion"]],indent=2,ensure_ascii=False)

    )

    try:
        response=llm_critic.invoke(messages)
    except Exception as e:
        raise RuntimeError(f"Critic Failed : {e}")
    
    return {

        "critic_feedback": response.content

    }

## Judge

In [16]:
from langchain.chat_models import init_chat_model

In [17]:
llm_judge=init_chat_model(
    model="qwen2.5:3b",
    model_provider="ollama"
)
llm_judge_struct=llm_judge.with_structured_output(JudgeDecision)
judge_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
            You are the judge of a multi-agent discussion.

            You must NEVER answer the user's question.

            You must NEVER summarize the discussion.

            Your only responsibility is to determine whether another discussion round is likely to meaningfully improve the quality of the final answer.

            Consider:

            - Has the discussion reached a well-reasoned conclusion?
            - Are important questions still unanswered?
            - Are there significant disagreements that require another round?
            - Does the critic identify meaningful improvements?
            - Is another discussion round likely to meaningfully improve the quality or correctness of the final answer?
            - Use the reviewer summary as a high-level overview, the critic feedback to identify remaining weaknesses, and the discussion history to verify important details.

            Rules:

            - If the discussion is already sufficient, stop.
            - If another round would add little value, stop.
            - If important issues remain unresolved, continue.
            - Be conservative when requesting another discussion round. Continue only if additional discussion is likely to produce meaningful improvements.
            - Minor stylistic issues or small disagreements alone are not sufficient reasons to continue the discussion.

            Return only the structured output with:

            - judge_approved: true if the discussion is sufficiently complete and no further discussion is likely to meaningfully improve the final answer.
            - judge_approved: false if another discussion round is likely to meaningfully improve the final answer.
            - judge_reason: a concise explanation supporting your decision based on the discussion quality.
        """
    ),

    (
        "human",
        """
            User Query:

            {query}

            Current Round:

            {round}

            Reviewer Summary:

            {review_summary}

            Critic Feedback:

            {critic_feedback}

            Discussion (JSON):

            {discussion}

            Each JSON object represents one discussion message.

            The discussion is ordered chronologically.
        """
    )

])
def judge(state:State):
    messages=judge_prompt.format_messages(
        query=state["query"],

        round=state["round"],

        review_summary=state["review_summary"],

        critic_feedback=state["critic_feedback"],

        discussion=json.dumps([msg.model_dump() for msg in state["discussion"]],indent=2,ensure_ascii=False)

    )

    try:
        decision=llm_judge_struct.invoke(messages)
    except Exception as e:
        raise RuntimeError(f"Judge Failed: {e}")
    
    return {

        "judge_approved": decision.judge_approved,

        "judge_reason": decision.judge_reason

    }

## Answer Node

In [18]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
import json

In [19]:
final_answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
            You are an expert AI assistant.

            Your responsibility is to generate the final answer to the user's query.

            The discussion has already been completed by multiple expert personas.

            Your task is to synthesize their reasoning into one clear, accurate, and well-structured response.

            Instructions:

            - Directly answer the user's question.
            - Ensure the final answer is complete and addresses all important aspects of the user's query.
            - Resolve conflicting viewpoints whenever possible.
            - If the discussion does not support a definitive conclusion, clearly explain the remaining uncertainty instead of guessing.
            - Do not mention personas.
            - Do not mention the reviewer.
            - Do not mention the critic.
            - Do not describe the discussion process.
            - Do not invent information not supported by the discussion.
            - Use the reviewer summary as the primary overview of the discussion.
            - If important alternative viewpoints remain valid, briefly acknowledge them before presenting the overall conclusion.
            - Use the discussion history to verify important details.
            - Ensure the final answer addresses important issues identified in the critic feedback whenever they are supported by the discussion.
            - Organize the answer logically with clear sections when appropriate.
            - Produce a natural, coherent answer.

        """
    ),

    (
        "human",
        """
            User Query:

            {query}

            Reviewer Summary:

            {review_summary}

            Critic Feedback:

            {critic_feedback}

            Discussion (JSON):

            {discussion}
        """
    )

])
llm_answer=init_chat_model(
    model="phi4-mini",
    model_provider="ollama"
)
def answer(state:State):
    messages=final_answer_prompt.format_messages(
        query=state["query"],

        review_summary=state["review_summary"],

        critic_feedback=state["critic_feedback"],

        discussion=json.dumps([msg.model_dump() for msg in state["discussion"]],indent=2,ensure_ascii=False)

    )

    try:
        response=llm_answer.invoke(messages)
    except Exception as e:
        raise RuntimeError(f"Final Answer Generation Failed: {e}")
    
    return {
        "final_answer": response.content
    }

## Final Summary Node

In [20]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
import json

In [21]:
final_summary_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
            You are an expert discussion summarizer.

            Your responsibility is to produce a concise and accurate summary of the completed multi-agent discussion.

            You must NEVER answer the user's question.

            Your goal is only to summarize how the discussion progressed and what conclusions were reached.

            Instructions:

            - Summarize the overall discussion in a clear and well-structured manner.
            - Highlight the major viewpoints discussed.
            - Mention important disagreements and how they were resolved, if applicable.
            - Mention any remaining uncertainties.
            - Use the reviewer summary as the primary overview of the discussion.
            - Use the discussion history to verify important details when necessary.
            - Do not invent information that was not discussed.
            - Do not mention personas by name. You may refer to their roles or perspectives when helpful.
            - Do not mention the reviewer.
            - Do not mention the critic.
            - Do not mention the judge.
            - Produce only the discussion summary.
        """
    ),
    (
        "human",
        """
            User Query:

            {query}

            Reviewer Summary:

            {review_summary}

            Discussion (JSON):

            {discussion}
        """
    )
])
llm_summary=init_chat_model(
    model="qwen2.5:3b",
    model_provider="ollama"
)
def summary(state:State):
    messages=final_summary_prompt.format_messages(
        query=state["query"],

        review_summary=state["review_summary"],

        discussion=json.dumps([msg.model_dump() for msg in state["discussion"]],indent=2,ensure_ascii=False)

    )

    try:
        response=llm_summary.invoke(messages)
    except Exception as e:
        raise RuntimeError(f"Final Discussion Summary Failed: {e}")
    
    return {
        "final_discussion_summary": response.content
    }

## Router Function

In [22]:
from typing import Literal

In [23]:
def judge_router(state: State) -> Literal["increment_round", "answer"]:
    
    # it Means judge Said that Discussion is Completed
    if state["judge_approved"]:
        return "answer"

    # It is For Safety Because It prevents Infinite Rounds
    if state["round"] >= state["max_rounds"]:
        return "answer"

    # It Means continue The Discussion
    return "increment_round"

## Round Update

In [24]:
def increment_round(state:State):
    print("---------------------",state["round"]+1,"--------------------------")
    return {
        "round":state["round"]+1
    }

## Graph Building

In [25]:
from langgraph.graph import StateGraph,START,END

In [26]:
graph_builder=StateGraph(State)


In [27]:
# graph_builder.add_node("planner", persona_planner)

# graph_builder.add_node("dispatcher", dispatcher)

# graph_builder.add_node("worker", worker)

# graph_builder.add_node("reviewer", reviewer)

# graph_builder.add_node("critic", critic)

# graph_builder.add_node("judge", judge)

# graph_builder.add_node("increment_round", increment_round)

# graph_builder.add_node("answer", answer)

# graph_builder.add_node("summary", summary)


In [28]:
graph_builder.add_node("planner", persona_planner)

graph_builder.add_node("worker", worker)

graph_builder.add_node("reviewer", reviewer)

graph_builder.add_node("critic", critic)

graph_builder.add_node("judge", judge)

graph_builder.add_node("increment_round", increment_round)

graph_builder.add_node("answer", answer)

graph_builder.add_node("summary", summary)

In [29]:
# graph_builder.add_edge(START, "planner")
# graph_builder.add_edge("planner", "dispatcher")
# graph_builder.add_conditional_edges("dispatcher",dispatcher)
# graph_builder.add_edge("worker", "reviewer")
# graph_builder.add_edge("reviewer", "critic")
# graph_builder.add_edge("critic", "judge")
# graph_builder.add_conditional_edges("judge",judge_router,{
#         "increment_round": "increment_round",
#         "answer": "answer",
# })
# graph_builder.add_edge("increment_round", "dispatcher")
# graph_builder.add_edge("answer", "summary")
# graph_builder.add_edge("summary", END)


In [30]:
graph_builder.add_edge(START, "planner")
graph_builder.add_conditional_edges("planner", dispatcher)
graph_builder.add_edge("worker", "reviewer")
graph_builder.add_edge("reviewer", "critic")
graph_builder.add_edge("critic", "judge")
graph_builder.add_conditional_edges("judge",judge_router,{
        "increment_round": "increment_round",
        "answer": "answer",
})
graph_builder.add_conditional_edges("increment_round",dispatcher)
graph_builder.add_edge("answer", "summary")
graph_builder.add_edge("summary", END)


## Memory

In [31]:
from langgraph.checkpoint.memory import InMemorySaver

In [32]:
memory=InMemorySaver()

## Compile The Graph

In [33]:
graph = graph_builder.compile()

## Graph Display

In [34]:
from IPython.display import display,Image

In [37]:
# display(Image(graph.get_graph().draw_mermaid_png()))

## Testing

In [38]:
initial_state = {
    "query": "Should AI replace human teachers?",
    "num_personas": 4,

    "personas": {},
    "discussion": [],

    "review_summary": "",
    "critic_feedback": "",

    "final_answer": "",
    "final_discussion_summary": "",

    "round": 1,
    "max_rounds": 3,

    "judge_approved": False,
    "judge_reason": "",
}

In [39]:
# result = graph.invoke(initial_state)
for event in graph.stream(initial_state):
    print(event)

{'planner': {'personas': {'persona1': RuntimePersona(id='persona1', name='Dr. Johnathan Smart', age=52, gender='Male', behavior='Analytical and objective', expertise='Educational Technology', goal='Ensure optimal learning outcomes with AI integration', personality='Determined, focused on efficiency gains', speaking_style='Technical, clear explanations of technical concepts', model=None, chat_history=[]), 'persona2': RuntimePersona(id='persona2', name='Dr. Elena Alvarez', age=48, gender='Female', behavior='Committed to human connection and empathy', expertise='Developmental Psychology', goal='Preserve the role of teachers in fostering emotional intelligence', personality='Passionate, emotionally intelligent', speaking_style='Emotional, emphasizes human touch', model=None, chat_history=[]), 'persona3': RuntimePersona(id='persona3', name='Mr. Michael Chen', age=45, gender='Male', behavior='Pragmatic and risk-averse', expertise='Business Administration', goal='Maximize cost-effectiveness a